# Codex v9

AC-MOT v9 ? Full 17-Sequence Evaluation, YOLOv8n Fair Comparison

Stage 1 compares only two systems:

- Baseline: YOLOv8n + ByteTrack
- Full AC-MOT: YOLOv8n + Scene Analysis + Adaptive Thresholding + Adaptive Resolution + ReID + Smart Calibration

The detector is fixed to YOLOv8n for both systems, so any change in MOTA, IDF1, Recall, HOTA, IDS, and FPS reflects the adaptive tracking architecture rather than detector capacity.

A separate optional ablation cell is included at the end and is disabled by default.


In [ ]:
# ????????????????????????????????????????????????????????
# ?  CELL 1 ? SETUP + FULL 17-SEQUENCE EVALUATION             ?
# ????????????????????????????????????????????????????????
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml -q

import os, time, shutil, gc, yaml
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import torch
import motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try:
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_acmot_v9_tmp')

assert SEQ_DIR.exists() and ANNOT_DIR.exists(), 'Dataset path not found'
all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])

# Representative quick set: crowded/low-altitude, long crowded, night/darker if available.
PREFERRED_3 = ['uav0000009_03358_v', 'uav0000077_00720_v', 'uav0000119_02301_v']
by_name = {s.name: s for s in all_sequences}
sequences = [by_name[n] for n in PREFERRED_3 if n in by_name]
if len(sequences) < 3:
    sequences = all_sequences[:3]

RUN_OPTIONAL_ABLATION = False
MODEL_NAME = 'yolov8n.pt'
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
HALF = DEVICE != 'cpu'

print(f'Codex v9 ready | Detector fixed: {MODEL_NAME} | Device={DEVICE} | FP16={HALF}')
print(f'Total sequences: {len(all_sequences)}')
print('Validation sequences:')
for s in sequences:
    print('  -', s.name)
print('Optional ablation enabled:', RUN_OPTIONAL_ABLATION)


In [ ]:
# ????????????????????????????????????????????????????????
# ?  CELL 2 ? METRICS + AC-MOT MODULES                  ?
# ????????????????????????????????????????????????????????

@dataclass
class SceneState:
    sci: float = 0.0
    scene: str = 'clear'
    brightness: float = 128.0
    blur: float = 500.0
    edge_density: float = 0.0
    crowd: float = 0.0
    tiny_ratio: float = 0.0
    n_dets: int = 0


class SceneAnalyzer:
    """Fast UAV-calibrated scene analysis for adaptive tracking decisions."""
    def __init__(self, window=5):
        self.sci_hist = deque(maxlen=window)

    def analyze(self, img: np.ndarray, prev_boxes: np.ndarray) -> SceneState:
        small = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        edge_density = float(cv2.Canny(gray, 50, 120).mean() / 255.0)
        n_dets = len(prev_boxes)
        crowd = min(n_dets / 25.0, 1.0)

        if n_dets:
            areas = (prev_boxes[:, 2] - prev_boxes[:, 0]) * (prev_boxes[:, 3] - prev_boxes[:, 1])
            tiny_ratio = float(np.mean(areas < 32 * 32))
        else:
            tiny_ratio = 0.0

        raw_sci = 0.35 * crowd + 0.25 * min(edge_density / 0.14, 1.0) + 0.25 * tiny_ratio
        if brightness < 80:
            raw_sci += 0.10
        if blur < 180:
            raw_sci += 0.05
        self.sci_hist.append(float(np.clip(raw_sci, 0.0, 1.0)))
        sci = float(np.mean(self.sci_hist))

        if brightness < 80:
            scene = 'night'
        elif blur < 180:
            scene = 'blur'
        elif tiny_ratio > 0.45:
            scene = 'tiny'
        elif crowd > 0.55 or edge_density > 0.10:
            scene = 'crowded'
        else:
            scene = 'clear'
        return SceneState(sci=sci, scene=scene, brightness=brightness, blur=blur,
                          edge_density=edge_density, crowd=crowd,
                          tiny_ratio=tiny_ratio, n_dets=n_dets)

    def reset(self):
        self.sci_hist.clear()


class SmartCalibrator:
    """Conservative calibration to improve recall without repeating the v5 MOTA collapse."""
    def __init__(self, adaptive_threshold=True, adaptive_resolution=True):
        self.adaptive_threshold = adaptive_threshold
        self.adaptive_resolution = adaptive_resolution

    def params(self, state: SceneState) -> dict:
        conf = 0.25
        iou = 0.45
        imgsz = 640
        if self.adaptive_threshold:
            conf = 0.245 - 0.055 * state.sci
            iou = 0.49 - 0.055 * state.sci
            if state.scene in ['crowded', 'tiny', 'night']:
                conf -= 0.015
            if state.scene == 'blur':
                iou -= 0.015
        if self.adaptive_resolution:
            if state.sci > 0.60 or state.tiny_ratio > 0.50:
                imgsz = 832
            elif state.sci > 0.35 or state.scene in ['crowded', 'tiny']:
                imgsz = 736
        return dict(conf=float(np.clip(conf, 0.17, 0.28)),
                    iou=float(np.clip(iou, 0.40, 0.52)),
                    imgsz=int(imgsz))


class LightweightReID:
    """Simple appearance-memory ID remapping for recently lost tracks."""
    def __init__(self, crop=24, bank=4, threshold=0.82, max_age=35):
        self.crop = crop
        self.bank_size = bank
        self.threshold = threshold
        self.max_age = max_age
        self.bank = defaultdict(lambda: deque(maxlen=bank))
        self.lost_feat = {}
        self.lost_age = {}
        self.seen_ids = set()
        self.frame_idx = 0

    def _feature(self, img: np.ndarray, box: np.ndarray):
        x1, y1, x2, y2 = [int(max(0, v)) for v in box]
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
        feat = cv2.resize(gray, (self.crop, self.crop)).ravel().astype(np.float32)
        feat = feat - feat.mean()
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 1e-6 else None

    def update_and_remap(self, img: np.ndarray, ids: np.ndarray, boxes: np.ndarray) -> np.ndarray:
        self.frame_idx += 1
        current = set(ids.tolist()) if len(ids) else set()
        remapped = ids.copy()

        for i, tid in enumerate(ids):
            tid = int(tid)
            feat = self._feature(img, boxes[i])
            if feat is None:
                continue
            is_new = tid not in self.seen_ids
            if is_new and self.lost_feat:
                best_tid, best_sim = tid, self.threshold
                for old_tid, old_feat in list(self.lost_feat.items()):
                    sim = float(np.dot(feat, old_feat))
                    if sim > best_sim:
                        best_tid, best_sim = old_tid, sim
                if best_tid != tid:
                    remapped[i] = best_tid
                    self.lost_feat.pop(best_tid, None)
                    self.lost_age.pop(best_tid, None)
                    tid = best_tid
            self.bank[tid].append(feat)
            self.seen_ids.add(tid)

        for tid in list(self.seen_ids):
            if tid not in current and tid not in self.lost_feat and self.bank[tid]:
                mean_feat = np.mean(np.stack(self.bank[tid]), axis=0)
                norm = np.linalg.norm(mean_feat)
                if norm > 1e-6:
                    self.lost_feat[tid] = mean_feat / norm
                    self.lost_age[tid] = self.frame_idx

        for tid, age in list(self.lost_age.items()):
            if self.frame_idx - age > self.max_age:
                self.lost_feat.pop(tid, None)
                self.lost_age.pop(tid, None)
        return remapped


def load_gt(path: Path) -> pd.DataFrame:
    """Load VisDrone MOT ground truth and keep evaluated vehicle/person classes."""
    if not path.exists():
        return pd.DataFrame()
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df = pd.read_csv(path, header=None, names=cols)
    df = df[df['cat'].isin([1,4,5,6,9])]
    df = df[(df['occ'] < 2) & (df['trunc'] < 2) & (df['score'] == 1)]
    return df.reset_index(drop=True)


def iou_dist(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """Return MOTMetrics distance matrix with shape gt x pred."""
    if not len(pred) or not len(gt):
        return np.empty((len(gt), len(pred)))
    ix1 = np.maximum(pred[:, 0:1].T, gt[:, 0:1])
    iy1 = np.maximum(pred[:, 1:2].T, gt[:, 1:2])
    ix2 = np.minimum(pred[:, 2:3].T, gt[:, 2:3])
    iy2 = np.minimum(pred[:, 3:4].T, gt[:, 3:4])
    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
    ap = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    ag = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    union = ap[np.newaxis, :] + ag[:, np.newaxis] - inter
    return 1.0 - np.where(union > 0, inter / union, 0.0)


def hota_approx(tp: int, fp: int, fn: int, ids: int) -> float:
    """Approximate HOTA for quick academic comparison without TrackEval."""
    det_a = tp / max(tp + fp + fn, 1)
    ass_a = max(0.0, 1.0 - ids / max(tp, 1))
    return float(np.sqrt(det_a * ass_a))


def eval_acc(acc, name='seq') -> dict:
    mh = mm.metrics.create()
    summ = mh.compute(acc,
        metrics=['mota','idf1','num_switches','recall','precision',
                 'num_misses','num_false_positives','num_matches'],
        name=name)
    row = summ.iloc[0]
    return dict(
        mota=float(row['mota']),
        idf1=float(row['idf1']),
        recall=float(row['recall']),
        precision=float(row['precision']),
        ids=int(row['num_switches']),
        fn=int(row['num_misses']),
        fp=int(row['num_false_positives']),
        matches=int(row['num_matches']),
        hota=hota_approx(int(row['num_matches']), int(row['num_false_positives']),
                         int(row['num_misses']), int(row['num_switches'])),
    )


def build_tracker_yaml(name: str, high: float, low: float, new: float, buffer: int, match: float) -> str:
    """Create a reproducible ByteTrack config for one experiment mode."""
    path = Path(f'/content/{name}.yaml')
    data = dict(tracker_type='bytetrack', track_high_thresh=float(high),
                track_low_thresh=float(low), new_track_thresh=float(new),
                track_buffer=int(buffer), match_thresh=float(match), fuse_score=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
    return str(path)


def reset_tracker(model):
    """Reset Ultralytics predictor so each sequence starts with clean tracker state."""
    if getattr(model, 'predictor', None) is not None:
        model.predictor = None

print('AC-MOT modules and metrics ready')


In [ ]:
# ????????????????????????????????????????????????????????
# ?  CELL 3 ? SHARED RUNNER                             ?
# ????????????????????????????????????????????????????????

TRACKERS = {
    'baseline': 'bytetrack.yaml',
    'acmot': build_tracker_yaml('bytetrack_v9_acmot', high=0.18, low=0.04, new=0.18, buffer=45, match=0.86),
}

SYSTEMS = [
    dict(name='Baseline_YOLOv8n_ByteTrack', model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False, scene_analysis=False,
         reid=False, smart_calibration=False),
    dict(name='Full_ACMOT_YOLOv8n', model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True, adaptive_resolution=True, scene_analysis=True,
         reid=True, smart_calibration=True),
]


def config_params(system: dict, state: SceneState) -> dict:
    """Return frame-level params for baseline, ablation stages, or full AC-MOT."""
    if system.get('smart_calibration', False):
        return SmartCalibrator(system['adaptive_threshold'], system['adaptive_resolution']).params(state)

    conf, iou, imgsz = 0.25, 0.45, 640
    if system.get('adaptive_threshold', False):
        if state.scene in ['crowded', 'tiny', 'night']:
            conf, iou = 0.205, 0.465
        elif state.scene == 'blur':
            conf, iou = 0.225, 0.455
        else:
            conf, iou = 0.255, 0.485
    if system.get('adaptive_resolution', False):
        if state.scene in ['crowded', 'tiny'] or state.sci > 0.45:
            imgsz = 736
        if state.sci > 0.65 or state.tiny_ratio > 0.55:
            imgsz = 832
    return dict(conf=float(conf), iou=float(iou), imgsz=int(imgsz))


def run_system(system: dict, seqs, run_tag: str):
    """Run one system over the selected validation sequences."""
    model = YOLO(system['model'])
    if HALF:
        model.model.half()

    rows = []
    scene_global = Counter()
    for seq in tqdm(seqs, desc=system['name']):
        gt = load_gt(ANNOT_DIR / f'{seq.name}.txt')
        frames_drive = sorted(seq.glob('*.jpg'))
        if gt.empty or not frames_drive:
            continue

        LOCAL_TMP.mkdir(exist_ok=True)
        local_seq = LOCAL_TMP / seq.name
        if local_seq.exists():
            shutil.rmtree(local_seq)
        shutil.copytree(seq, local_seq)
        frames = sorted(local_seq.glob('*.jpg'))

        reset_tracker(model)
        analyzer = SceneAnalyzer()
        reid = LightweightReID() if system.get('reid', False) else None
        acc = mm.MOTAccumulator(auto_id=True)
        times = []
        prev_boxes = np.empty((0, 4))
        state = SceneState()
        scene_counts = Counter()
        imgsz_seen, conf_seen = [], []

        for idx, frame_path in enumerate(frames, start=1):
            t0 = time.perf_counter()
            img = cv2.imread(str(frame_path))
            if img is None:
                continue

            if system.get('scene_analysis', False) and (idx == 1 or idx % 10 == 1):
                state = analyzer.analyze(img, prev_boxes)
            elif not system.get('scene_analysis', False):
                state = SceneState()
            scene_counts[state.scene] += 1
            scene_global[state.scene] += 1

            params = config_params(system, state)
            imgsz_seen.append(params['imgsz'])
            conf_seen.append(params['conf'])

            res = model.track(
                source=img,
                tracker=TRACKERS[system['tracker']],
                conf=params['conf'],
                iou=params['iou'],
                imgsz=params['imgsz'],
                half=HALF,
                persist=True,
                verbose=False,
                device=DEVICE,
            )
            times.append(time.perf_counter() - t0)

            if res[0].boxes.id is not None:
                pred_ids = res[0].boxes.id.cpu().numpy().astype(int)
                pred_boxes = res[0].boxes.xyxy.cpu().numpy()
            else:
                pred_ids = np.array([], dtype=int)
                pred_boxes = np.empty((0, 4))
            prev_boxes = pred_boxes.copy()

            if reid is not None and len(pred_ids):
                pred_ids = reid.update_and_remap(img, pred_ids, pred_boxes)

            gt_f = gt[gt['frame'] == idx]
            gt_ids = gt_f['id'].values
            gt_boxes = (np.column_stack([
                gt_f['x'].values,
                gt_f['y'].values,
                gt_f['x'].values + gt_f['w'].values,
                gt_f['y'].values + gt_f['h'].values,
            ]) if len(gt_f) else np.empty((0, 4)))

            dist = iou_dist(pred_boxes, gt_boxes)
            acc.update(gt_ids, pred_ids,
                       dist if dist.size else np.empty((len(gt_ids), len(pred_ids))))

        shutil.rmtree(local_seq, ignore_errors=True)
        metrics = eval_acc(acc, seq.name)
        fps = 1.0 / np.mean(times) if times else 0.0
        dom = scene_counts.most_common(1)[0][0] if scene_counts else 'unknown'
        rows.append(dict(
            run_tag=run_tag,
            system=system['name'],
            sequence=seq.name,
            frames=len(frames),
            fps=round(fps, 2),
            dominant_scene=dom,
            mean_imgsz=round(float(np.mean(imgsz_seen)), 1) if imgsz_seen else 640,
            mean_conf=round(float(np.mean(conf_seen)), 4) if conf_seen else 0.25,
            **metrics,
        ))
        tqdm.write(f"{system['name']:<24} {seq.name[:24]:24s} MOTA={metrics['mota']:.3f} IDF1={metrics['idf1']:.3f} HOTA={metrics['hota']:.3f} R={metrics['recall']:.3f} IDS={metrics['ids']} FPS={fps:.1f}")

    if HALF:
        torch.cuda.empty_cache()
    gc.collect()
    return pd.DataFrame(rows), scene_global


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Create academic-friendly aggregate table."""
    rows = []
    for system, g in df.groupby('system', sort=False):
        rows.append(dict(
            system=system,
            sequences=len(g),
            mota=g['mota'].mean(),
            idf1=g['idf1'].mean(),
            recall=g['recall'].mean(),
            precision=g['precision'].mean(),
            hota=g['hota'].mean(),
            ids=int(g['ids'].sum()),
            fn=int(g['fn'].sum()),
            fp=int(g['fp'].sum()),
            matches=int(g['matches'].sum()),
            fps=g['fps'].mean(),
            mean_imgsz=g['mean_imgsz'].mean(),
        ))
    out = pd.DataFrame(rows)
    if len(out) >= 2:
        base = out.iloc[0]
        for col in ['mota','idf1','recall','precision','hota','fps']:
            out[col + '_delta_vs_baseline'] = out[col] - float(base[col])
        out['ids_delta_vs_baseline'] = out['ids'] - int(base['ids'])
        out['fn_delta_vs_baseline'] = out['fn'] - int(base['fn'])
        out['fp_delta_vs_baseline'] = out['fp'] - int(base['fp'])
    return out

print('Runner ready: baseline vs Full AC-MOT uses YOLOv8n only')


In [ ]:
# ????????????????????????????????????????????????????????
# ?  CELL 4 ? FULL 17-SEQUENCE EVALUATION              ?
# ????????????????????????????????????????????????????????

import gc
import torch

torch.cuda.empty_cache()
gc.collect()

ts = datetime.now().strftime('%Y%m%d_%H%M%S')

run_tag = f'acmot_v9_full17_{ts}'

validation_rows = []

for system in SYSTEMS:

    print(f'\nRunning FULL evaluation for: {system["name"]}')

    df_sys, scene_counts = run_system(
        system,
        sequences,
        run_tag
    )

    validation_rows.append(df_sys)

validation_df = pd.concat(validation_rows, ignore_index=True)

validation_summary = summarize(validation_df)

print('\n' + '=' * 120)
print('AC-MOT v9 FULL EVALUATION — YOLOv8n FAIR COMPARISON, 17 SEQUENCES')
print('=' * 120)

main_cols = [
    'system',
    'sequences',
    'mota',
    'idf1',
    'recall',
    'hota',
    'ids',
    'fps',
    'fn',
    'fp',
    'mean_imgsz'
]

print(
    validation_summary[main_cols].to_string(
        index=False,
        float_format=lambda x: f'{x:.4f}'
    )
)

print('\nDeltas vs baseline:')

delta_cols = [
    'system',
    'mota_delta_vs_baseline',
    'idf1_delta_vs_baseline',
    'recall_delta_vs_baseline',
    'hota_delta_vs_baseline',
    'ids_delta_vs_baseline',
    'fps_delta_vs_baseline'
]

print(
    validation_summary[delta_cols].to_string(
        index=False,
        float_format=lambda x: f'{x:+.4f}'
    )
)

print('=' * 120)

print('\nPer-sequence results:')

seq_cols = [
    'system',
    'sequence',
    'mota',
    'idf1',
    'recall',
    'hota',
    'ids',
    'fps',
    'fn',
    'fp',
    'dominant_scene',
    'mean_imgsz',
    'mean_conf'
]

print(
    validation_df[seq_cols].to_string(
        index=False,
        float_format=lambda x: f'{x:.4f}'
    )
)

summary_path = DRIVE_RESULTS / f'{run_tag}_summary.csv'
seq_path = DRIVE_RESULTS / f'{run_tag}_per_sequence.csv'

validation_summary.to_csv(summary_path, index=False)
validation_df.to_csv(seq_path, index=False)

print(f'\nSaved summary CSV -> {summary_path}')
print(f'Saved per-sequence CSV -> {seq_path}')

base = validation_summary.iloc[0]
full = validation_summary.iloc[1]

print('\nAcademic interpretation helper:')

print(f"MOTA delta : {full['mota'] - base['mota']:+.4f}")
print(f"IDF1 delta : {full['idf1'] - base['idf1']:+.4f}")
print(f"Recall delta: {full['recall'] - base['recall']:+.4f}")
print(f"HOTA delta : {full['hota'] - base['hota']:+.4f}")
print(f"IDS delta  : {int(full['ids'] - base['ids']):+d} lower is better")
print(f"FPS delta  : {full['fps'] - base['fps']:+.2f}")

print('\nFinal verdict:')

if full['mota'] > base['mota'] and full['idf1'] > base['idf1']:
    print('AC-MOT full system shows measurable contribution over the baseline.')
else:
    print('The adaptive pipeline still needs further optimization.')


In [ ]:
# ????????????????????????????????????????????????????????
# ?  CELL 5 ? OPTIONAL 3-SEQUENCE ABLATION STUDY        ?
# ????????????????????????????????????????????????????????

if RUN_OPTIONAL_ABLATION:
    ABLATION_SYSTEMS = [
        dict(name='A0_Baseline', model=MODEL_NAME, tracker='baseline',
             adaptive_threshold=False, adaptive_resolution=False, scene_analysis=False,
             reid=False, smart_calibration=False),
        dict(name='A1_AdaptiveThreshold', model=MODEL_NAME, tracker='acmot',
             adaptive_threshold=True, adaptive_resolution=False, scene_analysis=True,
             reid=False, smart_calibration=False),
        dict(name='A2_AdaptiveResolution', model=MODEL_NAME, tracker='acmot',
             adaptive_threshold=True, adaptive_resolution=True, scene_analysis=True,
             reid=False, smart_calibration=False),
        dict(name='A3_ReID', model=MODEL_NAME, tracker='acmot',
             adaptive_threshold=True, adaptive_resolution=True, scene_analysis=True,
             reid=True, smart_calibration=False),
        dict(name='A4_Full_ACMOT', model=MODEL_NAME, tracker='acmot',
             adaptive_threshold=True, adaptive_resolution=True, scene_analysis=True,
             reid=True, smart_calibration=True),
    ]

    ts_abl = datetime.now().strftime('%Y%m%d_%H%M%S')
    abl_tag = f'acmot_v9_ablation_3seq_{ts_abl}'
    abl_rows = []
    for system in ABLATION_SYSTEMS:
        df_sys, _ = run_system(system, VAL_SEQS, abl_tag)
        abl_rows.append(df_sys)

    ablation_df = pd.concat(abl_rows, ignore_index=True)
    ablation_summary = summarize(ablation_df)

    print('\n' + '=' * 118)
    print('AC-MOT v9 OPTIONAL ABLATION ? 3 SEQUENCES, YOLOv8n FIXED')
    print('=' * 118)
    cols = ['system','sequences','mota','idf1','recall','hota','ids','fps','fn','fp','mean_imgsz']
    print(ablation_summary[cols].to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print('\nDeltas vs baseline:')
    delta_cols = ['system','mota_delta_vs_baseline','idf1_delta_vs_baseline','recall_delta_vs_baseline',
                  'hota_delta_vs_baseline','ids_delta_vs_baseline','fps_delta_vs_baseline']
    print(ablation_summary[delta_cols].to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
    print('=' * 118)

    abl_summary_path = DRIVE_RESULTS / f'{abl_tag}_summary.csv'
    abl_seq_path = DRIVE_RESULTS / f'{abl_tag}_per_sequence.csv'
    ablation_summary.to_csv(abl_summary_path, index=False)
    ablation_df.to_csv(abl_seq_path, index=False)
    print(f'Saved ablation summary CSV -> {abl_summary_path}')
    print(f'Saved ablation per-sequence CSV -> {abl_seq_path}')
else:
    print('Optional ablation is disabled.')
    print('To run it, set RUN_OPTIONAL_ABLATION = True in Cell 1 and rerun Cells 1-5.')
